# Phase 1 — Round-0 generation (chat-capable)
Generates initial patches for a base **or** instruct model. Instruct models use their chat template automatically. Upload `test_functions.csv` and `fewshot_pool.json` as a Kaggle dataset. Run: **Save Version -> Save & Run All (Commit)**.

In [ ]:
# ── CONFIG — edit this cell ─────────────────────────────────────────────
# Phase 1: round-0 generation. For the instruct model use the *-instruct id.
MODEL_NAME  = 'deepseek-ai/deepseek-coder-1.3b-instruct'
MODEL_ALIAS = 'deepseek_1.3b_instruct'

NUM_SAMPLES       = 5
MAX_NEW_TOKENS    = 512
TEMPERATURE       = 0.8
MAX_PROMPT_TOKENS = 1600
HF_TOKEN          = ''            # deepseek models are public; leave blank

USE_CHAT = 'instruct' in MODEL_NAME.lower()   # instruct -> chat template; base -> completion
OUTPUT_FILE = f'/kaggle/working/patches_raw_{MODEL_ALIAS}.jsonl'
print(f'Model {MODEL_NAME} | alias {MODEL_ALIAS} | chat={USE_CHAT} | '
      f'{NUM_SAMPLES} samples x {MAX_NEW_TOKENS} tok')

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('GPU' if device.type == 'cuda' else 'NO GPU — enable in Settings -> Accelerator')

In [ ]:
import glob, os, shutil, json
matches = glob.glob('/kaggle/input/**/test_functions.csv', recursive=True)
assert matches, 'Upload test_functions.csv as a dataset (+ Add Input).'
TEST_CSV_PATH = matches[0]
fs = glob.glob('/kaggle/input/**/fewshot_pool.json', recursive=True)
FEWSHOT_DATA = json.load(open(fs[0], encoding='utf-8')) if fs else {}
print(f'test data: {TEST_CSV_PATH} | fewshot pool: {"yes" if fs else "MISSING (few_shot -> zero_shot)"}')
partial = glob.glob(f'/kaggle/input/**/*{MODEL_ALIAS}*.jsonl', recursive=True)
if partial:
    shutil.copy(partial[0], OUTPUT_FILE)
    print(f'Resuming from {partial[0]} ({sum(1 for _ in open(OUTPUT_FILE))} lines)')

In [ ]:
import re

def core_zero_shot(fb, cwe):
    return (f'The following C/C++ function contains a {cwe} vulnerability.\n'
            f'Rewrite the function to fix the vulnerability.\n'
            f'Return ONLY the fixed function inside a ```c code block, no explanation.\n\n'
            f'Vulnerable function:\n```c\n{fb}\n```')

def core_few_shot(fb, cwe, pool):
    ex = pool.get(cwe, [])[:2]
    if not ex:
        return core_zero_shot(fb, cwe)
    s = f'Here are examples of fixing {cwe} vulnerabilities in C/C++.\n\n'
    for i, e in enumerate(ex, 1):
        s += (f'### Example {i}\nVulnerable:\n```c\n{e["func_before"]}\n```\n'
              f'Fixed:\n```c\n{e["func_after"]}\n```\n\n')
    s += (f'### Now fix this function\nVulnerable:\n```c\n{fb}\n```\n'
          f'Return ONLY the fixed function inside a ```c code block.')
    return s

def core_cot(fb, cwe):
    return (f'The C/C++ function below contains a {cwe} vulnerability.\n'
            f'Identify where the vulnerability occurs, what causes it, and what change fixes it.\n'
            f'Then write the complete corrected function inside a ```c code block.\n\n'
            f'Vulnerable function:\n```c\n{fb}\n```')

def extract_code(text):
    """Pull the fixed function out of a model response (chat or completion)."""
    m = re.search(r'```(?:c|cpp|c\+\+)?\s*\n(.*?)```', text, re.DOTALL)
    if m and m.group(1).strip():
        return m.group(1).strip()
    m2 = re.search(r'^(.*?)```', text, re.DOTALL)   # completion: code before first fence
    if m2 and m2.group(1).strip():
        return m2.group(1).strip()
    return text.strip()

print('Prompt builders loaded. Chat mode =', USE_CHAT)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Loading {MODEL_NAME} ...')
load_kwargs = dict(torch_dtype=torch.float16, device_map='auto')
tok_kwargs  = {}
if HF_TOKEN:
    load_kwargs['token'] = HF_TOKEN; tok_kwargs['token'] = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **tok_kwargs)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = 'left'

if USE_CHAT and tokenizer.chat_template is None:
    raise RuntimeError('USE_CHAT is True but this tokenizer has no chat template.')
print(f'Model loaded on {next(model.parameters()).device}  | chat mode = {USE_CHAT}')


In [ ]:
def generate_one(core_text):
    """Generate a completion for one prompt core, honouring chat vs completion mode.
    Renders the chat template to a STRING first (robust across transformers versions),
    then tokenises; add_special_tokens=False for chat avoids a double BOS."""
    if USE_CHAT:
        text = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': core_text}],
            tokenize=False, add_generation_prompt=True)
        enc = tokenizer(text, return_tensors='pt', truncation=True,
                        max_length=MAX_PROMPT_TOKENS, add_special_tokens=False).to(device)
    else:
        enc = tokenizer(core_text + '\n\nFixed function:\n```c\n',
                        return_tensors='pt', truncation=True,
                        max_length=MAX_PROMPT_TOKENS).to(device)
    input_len = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                             temperature=TEMPERATURE, top_p=0.95,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)


In [ ]:
import pandas as pd, time
df = pd.read_csv(TEST_CSV_PATH, encoding='utf-8')
print(f'Test functions: {len(df)}')

builders = {
    'zero_shot': lambda fb, cwe: core_zero_shot(fb, cwe),
    'few_shot':  lambda fb, cwe: core_few_shot(fb, cwe, FEWSHOT_DATA),
    'cot':       lambda fb, cwe: core_cot(fb, cwe),
}

def load_done(path):
    done = set()
    try:
        for line in open(path, encoding='utf-8'):
            line = line.strip()
            if line:
                r = json.loads(line); done.add((r['func_id'], r['prompt_type'], r['sample_idx']))
    except FileNotFoundError:
        pass
    return done

done = load_done(OUTPUT_FILE)
total = len(df) * len(builders) * NUM_SAMPLES
print(f'total calls {total} | done {len(done)}')
t0 = time.time()
with open(OUTPUT_FILE, 'a', encoding='utf-8') as out:
    for i, row in df.iterrows():
        for pt, b in builders.items():
            core = b(row['func_before'], row['cwe_id'])
            for s in range(NUM_SAMPLES):
                if (row['func_id'], pt, s) in done:
                    continue
                try:
                    raw = generate_one(core)
                    rec = {'model_alias': MODEL_ALIAS, 'func_id': row['func_id'],
                           'cwe_id': row['cwe_id'], 'prompt_type': pt, 'sample_idx': s,
                           'patch': extract_code(raw), 'generated_raw': raw,
                           'func_before': row['func_before'], 'func_after': row['func_after']}
                    out.write(json.dumps(rec, ensure_ascii=False) + '\n'); out.flush()
                    done.add((row['func_id'], pt, s))
                except RuntimeError as e:
                    if 'out of memory' in str(e).lower():
                        torch.cuda.empty_cache(); print('OOM skip', row['func_id'], pt, s)
                    else:
                        raise
        el = (time.time() - t0) / 60
        print(f'[{i+1}/{len(df)}] {row["func_id"]} done={len(done)} elapsed={el:.1f}m')
print(f'\nDONE -> {OUTPUT_FILE}')
print('Download it from the Output tab; validate locally with revalidate/validate_patches.')